# MarketPulse — PySpark ETL

Cleans raw tables and engineers features for downstream SQL analytics, statistics, A/B testing, and churn modeling.

Cleaning rules are based on findings from `01_data_ingestion.ipynb` (see local repo).

## 1. Setup — Load Raw Tables from S3

Reconnects to S3 using boto3 and loads all 8 raw tables into Spark DataFrames for processing.

In [0]:
import boto3
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F

s3 = boto3.client(
    "s3",
    aws_access_key_id="ACCESS_KEY_ID",
    aws_secret_access_key="SECRET_ACCESS_KEY",
    region_name="ap-southeast-2"
)

BUCKET = "marketpulse-narjeena-2026"

files = {
    "customers": "raw/customers/olist_customers_dataset.csv",
    "orders": "raw/orders/olist_orders_dataset.csv",
    "order_items": "raw/order_items/olist_order_items_dataset.csv",
    "products": "raw/products/olist_products_dataset.csv",
    "payments": "raw/payments/olist_order_payments_dataset.csv",
    "reviews": "raw/reviews/olist_order_reviews_dataset.csv",
    "sessions": "raw/sessions/sessions.csv",
    "experiments": "raw/experiments/experiments.csv",
}

dfs = {}
for name, key in files.items():
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    pdf = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))
    dfs[name] = spark.createDataFrame(pdf)

customers = dfs["customers"]
orders = dfs["orders"]
order_items = dfs["order_items"]
products = dfs["products"]
payments = dfs["payments"]
reviews = dfs["reviews"]
sessions = dfs["sessions"]
experiments = dfs["experiments"]

print("All tables loaded.")

All tables loaded.


## 2. Clean Orders

Converts date columns from string to proper timestamp type, and adds an `is_delivered` boolean flag instead of filling null delivery dates — since a null delivery date is meaningful (order cancelled or still in transit), not an error to be patched over.

In [0]:
orders_clean = (
    orders
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn("is_delivered", F.col("order_status") == "delivered")
)

orders_clean.select(
    "order_id", "order_status", "is_delivered",
    "order_purchase_timestamp", "order_delivered_customer_date"
).show(5)

+--------------------+------------+------------+------------------------+-----------------------------+
|            order_id|order_status|is_delivered|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------+------------+------------------------+-----------------------------+
|e481f51cbdc54678b...|   delivered|        true|     2017-10-02 10:56:33|          2017-10-10 21:25:13|
|53cdb2fc8bc7dce0b...|   delivered|        true|     2018-07-24 20:41:37|          2018-08-07 15:27:45|
|47770eb9100c2d0c4...|   delivered|        true|     2018-08-08 08:38:49|          2018-08-17 18:06:29|
|949d5b44dbf5de918...|   delivered|        true|     2017-11-18 19:28:06|          2017-12-02 00:28:42|
|ad21c59c0840e6cb8...|   delivered|        true|     2018-02-13 21:18:39|          2018-02-16 18:17:02|
+--------------------+------------+------------+------------------------+-----------------------------+
only showing top 5 rows


## 3. Clean Products

Fills missing `product_category_name` with `"unknown"` rather than dropping those rows, to avoid silently losing revenue in later aggregations. Drops the 2 rows missing physical dimensions, since that's a trivial and safe loss.

In [0]:


products_clean = (
    products
    .fillna({"product_category_name": "unknown"})
    .dropna(subset=["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])
)

print(f"Original: {products.count()} rows, Cleaned: {products_clean.count()} rows")
products_clean.select("product_id", "product_category_name", "product_weight_g").show(5)

Original: 32951 rows, Cleaned: 32949 rows
+--------------------+---------------------+----------------+
|          product_id|product_category_name|product_weight_g|
+--------------------+---------------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|           225.0|
|3aa071139cb16b67c...|                artes|          1000.0|
|96bd76ec8810374ed...|        esporte_lazer|           154.0|
|cef67bcfe19066a93...|                bebes|           371.0|
|9dc1a7de274444849...| utilidades_domest...|           625.0|
+--------------------+---------------------+----------------+
only showing top 5 rows


## 4. Sanity Check — Verify Cleaning

Confirms the `is_delivered` flag and order status counts match what was found during initial exploration (see `01_data_ingestion.ipynb`).

In [0]:
orders_clean.groupBy("order_status", "is_delivered").count().orderBy(F.desc("count")).show()

+------------+------------+-----+
|order_status|is_delivered|count|
+------------+------------+-----+
|   delivered|        true|96478|
|     shipped|       false| 1107|
|    canceled|       false|  625|
| unavailable|       false|  609|
|    invoiced|       false|  314|
|  processing|       false|  301|
|     created|       false|    5|
|    approved|       false|    2|
+------------+------------+-----+



## 5. Build Unified Transactions Table

Joins orders, order_items, products, and payments into a single transaction-level table — this becomes the base for all downstream SQL analytics, RFM feature engineering, and the churn model. Only `delivered` orders are included, since cancelled/undelivered orders shouldn't count toward revenue or customer value metrics.

In [0]:
transactions = (
    orders_clean.filter(F.col("is_delivered") == True)
    .join(order_items, on="order_id", how="inner")
    .join(products_clean, on="product_id", how="inner")
    .join(
        payments.groupBy("order_id").agg(
            F.sum("payment_value").alias("total_payment_value"),
            F.max("payment_installments").alias("max_installments")
        ),
        on="order_id", how="left"
    )
)

print(f"Transactions row count: {transactions.count()}")
transactions.select(
    "order_id", "customer_id", "product_category_name",
    "price", "freight_value", "total_payment_value",
    "order_purchase_timestamp"
).show(5)

Transactions row count: 110179
+--------------------+--------------------+---------------------+-----+-------------+-------------------+------------------------+
|            order_id|         customer_id|product_category_name|price|freight_value|total_payment_value|order_purchase_timestamp|
+--------------------+--------------------+---------------------+-----+-------------+-------------------+------------------------+
|00010242fe8c5a6d1...|3ce436f183e68e078...|           cool_stuff| 58.9|        13.29|              72.19|     2017-09-13 08:59:02|
|00018f77f2f0320c5...|f6dd3ec061db4e398...|             pet_shop|239.9|        19.93|             259.83|     2017-04-26 10:53:06|
|000229ec398224ef6...|6489ae5e4333f3693...|     moveis_decoracao|199.0|        17.87|             216.87|     2018-01-14 14:33:31|
|00024acbcdf0a6daa...|d4eb9395c8c0431ee...|           perfumaria|12.99|        12.79|              25.78|     2018-08-08 10:00:35|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|   ferram

## 6. Customer-Level Feature Engineering (RFM)

Aggregates to one row per customer, computing Recency, Frequency, and Monetary value plus average delivery time and category diversity. These features feed the churn model and Power BI segmentation.

**Important:** grouped by `customer_unique_id`, not `customer_id`. Olist's `customer_id` is generated per-order, not per-person — grouping by it would mean every "customer" has frequency = 1 by construction, making repeat-purchase behavior undetectable. This was caught as a real bug during cohort retention analysis (see NOTES.md) and fixed here at the source.

In [0]:
order_level_unique = (
    transactions
    .select("order_id", "customer_id", "total_payment_value")
    .dropDuplicates(["order_id"])
    .join(customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
)

order_dates_unique = (
    orders_clean.filter(F.col("is_delivered") == True)
    .join(customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
    .select("order_id", "customer_unique_id", "order_purchase_timestamp")
    .dropDuplicates(["order_id"])
)

max_date = order_dates_unique.agg(F.max("order_purchase_timestamp")).collect()[0][0]

customer_features = (
    order_level_unique
    .join(order_dates_unique.select("order_id", "customer_unique_id", "order_purchase_timestamp"), on=["order_id", "customer_unique_id"])
    .groupBy("customer_unique_id")
    .agg(
        F.max("order_purchase_timestamp").alias("last_purchase_date"),
        F.countDistinct("order_id").alias("frequency"),
        F.sum("total_payment_value").alias("monetary_value"),
        F.avg("total_payment_value").alias("avg_order_value")
    )
    .withColumn("recency_days", F.datediff(F.lit(max_date), F.col("last_purchase_date")))
)


delivery_features_unique = (
    orders_clean.filter(F.col("is_delivered") == True)
    .join(customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .groupBy("customer_unique_id")
    .agg(F.avg("delivery_days").alias("avg_delivery_days"))
)

category_diversity_unique = (
    transactions
    .join(customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
    .groupBy("customer_unique_id")
    .agg(F.countDistinct("product_category_name").alias("distinct_categories"))
)

customer_features = (
    customer_features
    .join(delivery_features_unique, on="customer_unique_id", how="left")
    .join(category_diversity_unique, on="customer_unique_id", how="left")
)

print(f"Unique customers (by customer_unique_id): {customer_features.count()}")
customer_features.orderBy(F.desc("frequency")).show(5)

Unique customers (by customer_unique_id): 93345
+--------------------+-------------------+---------+------------------+------------------+------------+------------------+-------------------+
|  customer_unique_id| last_purchase_date|frequency|    monetary_value|   avg_order_value|recency_days| avg_delivery_days|distinct_categories|
+--------------------+-------------------+---------+------------------+------------------+------------+------------------+-------------------+
|8d50f5eadf50201cc...|2018-08-20 19:14:26|       15| 879.2699999999999| 58.61799999999999|           9|               4.2|                  4|
|3e43e6105506432c9...|2018-02-27 18:36:39|        9|1172.6599999999999|130.29555555555555|         183| 13.88888888888889|                  5|
|6469f99c1f9dfae77...|2018-06-28 00:43:34|        7| 758.8299999999999| 108.4042857142857|          62|               4.0|                  1|
|1b6c7548a2a1f9037...|2018-02-14 13:22:12|        7|            959.01|137.00142857142856|    

## 7. Save Processed Tables

Persists all cleaned/engineered tables as Delta tables, so every downstream notebook (SQL analytics, statistics, A/B testing, churn modeling) can query them directly without re-running this ETL pipeline.

In [0]:
transactions.write.format("delta").mode("overwrite").saveAsTable("marketpulse_transactions")
customer_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("marketpulse_customer_features")
orders_clean.write.format("delta").mode("overwrite").saveAsTable("marketpulse_orders_clean")
sessions.write.format("delta").mode("overwrite").saveAsTable("marketpulse_sessions")
experiments.write.format("delta").mode("overwrite").saveAsTable("marketpulse_experiments")
customers.write.format("delta").mode("overwrite").saveAsTable("marketpulse_customers")
payments.write.format("delta").mode("overwrite").saveAsTable("marketpulse_payments")
reviews.write.format("delta").mode("overwrite").saveAsTable("marketpulse_reviews")

print("All 8 tables saved.")

All 8 tables saved.


## Summary

This notebook took the raw Olist tables plus synthetic sessions/experiments data and produced **8 reusable Delta tables**:

- **marketpulse_orders_clean** — orders with proper datetime types and an `is_delivered` flag
- **marketpulse_transactions** — order-item level table joined with products and aggregated payments, filtered to delivered orders only
- **marketpulse_customer_features** — one row per real customer (`customer_unique_id`) with RFM features, average order value, average delivery time, and category diversity
- **marketpulse_customers** — raw customer table, including the `customer_unique_id` mapping
- **marketpulse_payments** — raw payments, including `payment_type` (used later as a voucher/discount proxy)
- **marketpulse_reviews** — raw reviews, used in statistical analysis
- **marketpulse_sessions** / **marketpulse_experiments** — synthetic tables for funnel and A/B testing analysis

**Two real bugs were caught and fixed during this process** (both documented in detail in NOTES.md):
1. Summing order-level payment totals directly from the item-level transactions table double-counted revenue for multi-item orders — fixed by deduplicating to one row per order before aggregating.
2. `customer_features` was originally grouped by `customer_id`, which is generated per-order in this dataset, making every customer appear as a one-time buyer by construction. Fixed by rebuilding the table grouped on `customer_unique_id`, the true persistent customer identifier.

**Next step:** `03_sql_analysis` — build the analytical SQL layer on top of these Delta tables.